# 01 — Data Exploration

Explore the raw market data: distributions, correlations, anomalies, and data quality.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.config import load_config
from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.universe import UniverseProvider

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# Load configuration and data
cfg = load_config('../config.yaml')
universe = UniverseProvider(cfg.data)
tickers = universe.get_tickers()
print(f'Universe: {len(tickers)} tickers')

loader = DataLoader(cfg.data)
raw_data = loader.load_universe(tickers)
prices = loader.build_price_matrix(raw_data)
volumes = loader.build_volume_matrix(raw_data)
print(f'Price matrix: {prices.shape}')

In [ ]:
# Data quality: missing values per ticker
missing_pct = prices.isna().mean().sort_values(ascending=False)
print(f'Tickers with >5% missing: {(missing_pct > 0.05).sum()}')
print(f'Overall NaN%: {prices.isna().mean().mean():.4%}')

fig, ax = plt.subplots(figsize=(14, 4))
missing_pct.plot.bar(ax=ax, color='steelblue')
ax.axhline(y=0.10, color='red', linestyle='--', label='10% threshold')
ax.set_title('Missing Data % per Ticker')
ax.set_ylabel('NaN Fraction')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verification: SPY close on 2020-03-23 should be ~$222
if 'SPY' in raw_data:
    spy = raw_data['SPY']
    target_date = '2020-03-23'
    if target_date in spy.index.strftime('%Y-%m-%d').values:
        spy_close = spy.loc[target_date, 'Close']
        print(f'SPY close on {target_date}: ${spy_close:.2f} (expected ~$222)')
    else:
        print(f'{target_date} not in SPY data')
else:
    print('SPY not in universe — add it for verification')

In [ ]:
# Clean data
cleaner = DataCleaner(cfg.data)
clean_prices, returns = cleaner.clean(prices, volumes)
print(f'Clean price matrix: {clean_prices.shape}')

In [ ]:
# Return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all daily returns
all_returns = returns.values.flatten()
all_returns = all_returns[~np.isnan(all_returns)]
axes[0].hist(all_returns, bins=200, density=True, alpha=0.7, color='steelblue')
axes[0].set_title('Daily Return Distribution (all stocks)')
axes[0].set_xlabel('Daily Return')
axes[0].set_xlim(-0.15, 0.15)

# QQ plot
from scipy import stats
sorted_ret = np.sort(all_returns)
theoretical = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_ret)))
axes[1].scatter(theoretical[::100], sorted_ret[::100], s=2, alpha=0.5)
axes[1].plot([-4, 4], [-4*np.std(all_returns), 4*np.std(all_returns)], 'r--')
axes[1].set_title('QQ Plot vs Normal')
axes[1].set_xlabel('Theoretical Quantiles')
axes[1].set_ylabel('Sample Quantiles')

plt.tight_layout()
plt.show()

print(f'Mean: {np.mean(all_returns):.6f}')
print(f'Std:  {np.std(all_returns):.6f}')
print(f'Skew: {stats.skew(all_returns):.3f}')
print(f'Kurt: {stats.kurtosis(all_returns):.3f}')

In [ ]:
# Cross-sectional correlation structure
corr = returns.tail(252).corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-0.5, vmax=1.0)
ax.set_title('Return Correlation Matrix (trailing 1 year)')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

# Average pairwise correlation
upper = corr.values[np.triu_indices_from(corr.values, k=1)]
print(f'Average pairwise correlation: {upper.mean():.3f}')
print(f'Median pairwise correlation:  {np.median(upper):.3f}')

In [ ]:
# Price history of selected stocks
sample_tickers = ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ'][:min(5, len(clean_prices.columns))]
if not sample_tickers:
    sample_tickers = clean_prices.columns[:5].tolist()

fig, ax = plt.subplots(figsize=(14, 6))
for ticker in sample_tickers:
    if ticker in clean_prices.columns:
        norm = clean_prices[ticker] / clean_prices[ticker].dropna().iloc[0]
        ax.semilogy(norm.index, norm.values, label=ticker, linewidth=1)
ax.set_title('Normalized Price History (log scale)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()